# ⚡ Parallelle agentarbeidsflyter med Microsoft Foundry (Python)

## 📋 Avansert veiledning for parallell behandling

Denne notatboken demonstrerer **parallelle arbeidsflytmønstre** ved bruk av Microsoft Agent Framework. Du vil lære hvordan du bygger høyytelses arbeidsflyter med parallell behandling der flere AI-agenter kjører samtidig, noe som dramatisk forbedrer gjennomstrømningen og muliggjør sofistikerte flertrådede forretningsprosesser.

> **Migrasjonsmerknad:** Dette eksemplet refererte tidligere til GitHub Models. GitHub Models er utfaset (løper ut juli 2026), så det bruker nå **Microsoft Foundry** gjennom `FoundryChatClient`, som bruker Azure OpenAI **Responses API**.

## 🎯 Læringsmål

### 🚀 **Grunnleggende for parallel behandling**
- **Parallell agentkjøring**: Kjør flere agenter samtidig for maksimal effektivitet
- **Orkestrering av arbeidsflyt**: Koordiner konkurrerende operasjoner samtidig som datakonsistens opprettholdes
- **Ytelsesoptimalisering**: Oppnå betydelig akselerasjon gjennom parallell behandling
- **Ressursstyring**: Effektiv utnyttelse av AI-modellressurser på tvers av samtidige operasjoner

### 🏗️ **Avanserte samtidighetsmønstre**
- **Fork-Join behandling**: Del opp arbeid på flere agenter og sammenstill resultater
- **Pipelining parallellisme**: Overlappende utføringsstadier for kontinuerlig gjennomstrømning
- **Lastbalansering**: Fordel arbeidet jevnt på tilgjengelige agentressurser
- **Synkroniseringspunkter**: Koordiner samtidige agenter i kritiske arbeidsflytstadier

### 🏢 **Samtidige bedriftsapplikasjoner**
- **Dokumentbehandling med høyt volum**: Behandle flere dokumenter samtidig
- **Analyse av innhold i sanntid**: Konkurrerende analyse av innkommende datastreams
- **Optimalisering av batch-behandling**: Maksimer gjennomstrømning for storskala operasjoner
- **Multimodal analyse**: Parallell behandling av ulike innholdstyper (tekst, bilder, data)

## ⚙️ Forutsetninger og oppsett

### 📦 **Nødvendige avhengigheter**

Installer Agent Framework med muligheter for parallelle arbeidsflyter:

```bash
pip install agent-framework -U
```

### 🔑 **Microsoft Foundry-konfigurasjon**

Logg inn med Azure CLI (`az login`) slik at `AzureCliCredential` kan autentisere, og sett deretter detaljene for Microsoft Foundry-prosjektet ditt.

**Oppsett av miljø (.env-fil):**
```env
AZURE_AI_PROJECT_ENDPOINT=https://<your-project>.services.ai.azure.com
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-5-mini
```

**Vurderinger ved parallell behandling:**
- **Ratelimiter**: Overvåk Azure OpenAI sine ratelimiter for samtidige forespørsler
- **Ressursbruk**: Vurder minne- og CPU-bruk ved flere samtidige agenter
- **Feilhåndtering**: Implementer robust feilgjenoppretting for parallelle operasjoner

### 🏗️ **Arkitektur for samtidige arbeidsflyter**

```mermaid
graph TD
    A[Arbeidsflyt Start] --> B[Samtidig Utførelse]
    B --> C[Agentpool 1]
    B --> D[Agentpool 2]
    B --> E[Agentpool 3]
    C --> F[Resultatsammenstilling]
    D --> F
    E --> F
    F --> G[Endelig Utdata]
    
    H[Microsoft Foundry] --> C
    H --> D
    H --> E
```

**Viktige fordeler:**
- **⚡ Ytelse**: Betydelig akselerasjon gjennom parallell kjøring
- **📈 Skalerbarhet**: Håndter økte arbeidsmengder uten proporsjonal tidsøkning
- **🔄 Effektivitet**: Bedre utnyttelse av tilgjengelige beregningsressurser
- **🎯 Gjennomstrømning**: Behandle mer arbeid på samme tid

## 🎨 **Designmønstre for samtidige arbeidsflyter**

### 🔍 **Forsknings- og analysepipeline**
```
Research Task → Parallel Research Agents → Content Synthesis → Quality Review
```

### 📊 **Data behandling arbeidsflyt**
```
Input Data → Concurrent Processing Agents → Result Aggregation → Final Report
```

### 🎭 **Innholdsskapingspipeline**
```
Content Brief → Parallel Content Generators → Review & Merge → Final Content
```

### 🔄 **Flerstegs behandling**
```
Input → Stage 1 (Concurrent) → Stage 2 (Concurrent) → Stage 3 (Sequential) → Output
```

## 🏢 **Bedriftsfordeler ved ytelse**

### ⚡ **Optimalisering av gjennomstrømning**
- **Parallell kjøring**: Flere agenter som jobber samtidig
- **Ressursutnyttelse**: Maksimal effektivitet av tilgjengelig AI-modellkapasitet
- **Tidsreduksjon**: Betydelig reduksjon i total behandlingstid
- **Skalerbar arkitektur**: Legg enkelt til flere samtidige agenter etter behov

### 🛡️ **Pålidelighet og robusthet**
- **Feiltoleranse**: Individuelle agentfeil stopper ikke hele arbeidsflyten
- **Feilisolasjon**: Problemer i én samtidig gren påvirker ikke andre
- **Grasiøs degradering**: Systemet fortsetter å fungere selv med redusert agentkapasitet
- **Gjenopprettingsmekanismer**: Automatisk nyforsøk og feilhåndtering for mislykkede operasjoner

### 📊 **Overvåking og observasjon**
- **Samtidig kjøringsovervåking**: Overvåk fremdriften for alle parallelle operasjoner
- **Ytelsesmetrikker**: Mål akselerasjon og effektivitetsgevinster
- **Ressursbruksanalyse**: Optimaliser tildeling av samtidige agenter
- **Flaskehalsidentifisering**: Finn og løs ytelsesbegrensninger

La oss bygge høyytelses samtidige AI-arbeidsflyter! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
import os
from typing import Any

from agent_framework import (
    Executor,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowViz,
    handler,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

load_dotenv()


In [ ]:
# Configure the Microsoft Foundry client with keyless authentication.
# FoundryChatClient targets the Azure OpenAI Responses API.
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


In [ ]:
ResearcherAgentName = "Researcher-Agent"
ResearcherAgentInstructions = "You are my travel researcher, working with me to analyze the destination, list relevant attractions, and make detailed plans for each attraction."

In [ ]:
PlanAgentName = "Plan-Agent"
PlanAgentInstructions = "You are my travel planner, working with me to create a detailed travel plan based on the researcher's findings."

In [ ]:
research_agent = provider.as_agent(
    name=ResearcherAgentName,
    instructions=ResearcherAgentInstructions,
)

plan_agent = provider.as_agent(
    name=PlanAgentName,
    instructions=PlanAgentInstructions,
)


In [ ]:
# A passthrough executor that broadcasts the user input to every agent in parallel.
class InputDispatcher(Executor):
    """Forward the user input unchanged to all participating agents."""

    @handler
    async def forward(self, text: str, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(text)


dispatcher = InputDispatcher(id="dispatcher")
agents = [research_agent, plan_agent]

workflow = (
    WorkflowBuilder(
        start_executor=dispatcher,
        output_executors=agents,
    )
    .add_fan_out_edges(dispatcher, agents)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")

In [ ]:
events = await workflow.run("Plan a trip to Seattle in December")
outputs = events.get_outputs()

In [ ]:
if outputs:
    print("===== Final Aggregated Responses =====")
    # outputs is a list of AgentResponse objects, one per output executor
    # (research_agent then plan_agent), in the order given to output_executors.
    for i, response in enumerate(outputs, start=1):
        print(f"{'-' * 60}\n\n{i:02d}:\n{response.text}")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokumentet er oversatt ved hjelp av AI-oversettelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selv om vi streber etter nøyaktighet, vær oppmerksom på at automatiske oversettelser kan inneholde feil eller unøyaktigheter. Det opprinnelige dokumentet på originalspråket skal betraktes som den autoritative kilden. For kritisk informasjon anbefales profesjonell menneskelig oversettelse. Vi er ikke ansvarlige for eventuelle misforståelser eller feiltolkninger som oppstår ved bruk av denne oversettelsen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
